In [1]:
import pandas as pd

import torch
from torch import nn

from kagglekit import (
    Dataset,
    LinearClassifier,
    Trainer,
    plot_histories,
    summarize_histories,
)

## Load Train/Val Split

In [2]:
data = torch.load('../data/train_val.pt')

X_train, X_val = data['X_train'], data['X_val']

Y_train = data['Y_train'].float().reshape(-1, 1)
Y_val = data['Y_val'].float().reshape(-1, 1)

X_train.shape, X_val.shape

(torch.Size([713, 20]), torch.Size([178, 20]))

### Majority Class Baseline

In [3]:
majority = Y_train.mean().round()
baseline_acc = (Y_val == majority).float().mean()

baseline_acc.item()

0.617977499961853

In [4]:
histories = {}

## Baseline Model: Linear Classifier

In [5]:
dataset = Dataset(X_train, Y_train, X_val, Y_val, batch_size=32)

In [6]:
torch.manual_seed(42)

model = LinearClassifier(in_features=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.SGD(model.parameters(), lr=0.01)

trainer = Trainer(loss_fn, optim, max_epochs=500)

history = trainer.fit(model, dataset)

histories['SGD-500-0.01'] = history

In [ ]:
torch.manual_seed(42)

model = LinearClassifier(in_features=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.SGD(model.parameters(), lr=0.001)

trainer = Trainer(loss_fn, optim, max_epochs=1500)

history = trainer.fit(model, dataset)

histories['SGD-1200-0.001'] = history

In [ ]:
torch.manual_seed(42)

model = LinearClassifier(in_features=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.01)

trainer = Trainer(loss_fn, optim, max_epochs=100)

history = trainer.fit(model, dataset)

histories['Adam-100-0.01'] = history

In [ ]:
torch.manual_seed(42)

model = LinearClassifier(in_features=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.01)

trainer = Trainer(loss_fn, optim, max_epochs=500)

history = trainer.fit(model, dataset)

histories['Adam-500-0.01'] = history

In [ ]:
fig = plot_histories(histories, baseline=baseline_acc.item())

In [ ]:
summarize_histories(histories).round(4)

## Load Full Train Set

In [ ]:
data = torch.load('../data/train_full.pt')

X = data['X']

Y = data['Y'].float().reshape(-1, 1)

X.shape, Y.shape

In [ ]:
dataset = Dataset(X, Y, batch_size=32)

### SGD, lr=0.01, epochs=136

In [ ]:
torch.manual_seed(42)

model = LinearClassifier(in_features=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.SGD(model.parameters(), lr=0.01)

trainer = Trainer(loss_fn, optim, max_epochs=136)

history = trainer.fit(model, dataset)

#### Submission

In [ ]:
test = torch.load('../data/test.pt')

X = test['X']
ids = test['PassengerId']

X.shape, ids.shape

In [ ]:
model.eval()

preds = model.predict(X.to(trainer.device)).cpu()

preds.shape

In [ ]:
submission = pd.DataFrame({
    'PassengerId': ids,
    'Survived': preds,
})
submission.to_csv('../submissions/linear_sgd_0.01_136.csv', index=False)

submission.head()

### Adam, lr=0.01, epochs=9

In [ ]:
torch.manual_seed(42)

model = LinearClassifier(in_features=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.01)

trainer = Trainer(loss_fn, optim, max_epochs=9)

history = trainer.fit(model, dataset)

#### Submission

In [ ]:
model.eval()

preds = model.predict(X.to(trainer.device)).cpu()

preds.shape

In [ ]:
submission = pd.DataFrame({
    'PassengerId': ids,
    'Survived': preds,
})
submission.to_csv('../submissions/linear_adam_0.01_09.csv', index=False)

submission.head()